In [12]:
from nlp4bia.datasets.Dataset import BenchmarkDataset
from nlp4bia.datasets import config
from nlp4bia.datasets.utils import handlers

import os
        
from requests import get
from zipfile import ZipFile
from io import BytesIO
import pandas as pd

class MeddoplaceLoader(BenchmarkDataset):
    URL = "https://zenodo.org/records/8403498/files/meddoplace_train+test+gazz+crossmap+multilingual_231003.zip?download=1"
    NAME = "meddoplace_train+test+gazz+crossmap+multilingual"
    DS_COLUMNS = config.DS_COLUMNS
    
    def __init__(self, lang="es", path=None, name=NAME, url=URL, download_if_missing=True):
        super().__init__(lang, name, path, url, download_if_missing)

    def load_data(self):
        '''Load the data from the dataset
        Output: DataFrame with columns: filename, mark, label, off0, off1, span, code, semantic_rel, split, text
        '''
        
        train_path = os.path.join(self.path, "meddoplace_train/tsv/meddoplace_tsv_train_complete.tsv")
        texts_train_path = os.path.join(self.path, "meddoplace_train/txt")
        test_path = os.path.join(self.path, "meddoplace_test/tsv/meddoplace_tsv_test_complete.tsv")
        texts_test_path = os.path.join(self.path, "meddoplace_test/txt")

        df_train = pd.read_csv(train_path, sep="\t", dtype=str)
        df_test = pd.read_csv(test_path, sep="\t", dtype=str)
        
        df_train["split"] = "train"
        df_test["split"] = "test"
        
        df = pd.concat([df_train, df_test])
        df.rename(columns={"text": "span"}, inplace=True)
        
        df_texts = handlers.get_texts(texts_train_path, texts_test_path)
        df = df.merge(df_texts, on="filename", how="left")
        
        self.df = df
        
        return df
    
    def preprocess_data(self):
        print("preprocessing data...")
        # DS_COLUMNS =  ["filenameid", "mention_class", "span", "code", "sem_rel", "is_abbreviation", "is_composite", "needs_context", "extension_esp"]
        
        d_map_names = {"label": "mention_class", "need_context": "needs_context"}
        span_pos_cols = ["start_span", "end_span"]
        
        self.df["filenameid"] = self.df["filename"] + "#" + self.df[span_pos_cols[0]] + "#" + self.df[span_pos_cols[1]]
        self.df.drop(columns=["filename"] + span_pos_cols, inplace=True)

        self.df.rename(columns=d_map_names, inplace=True)
        
        for col in self.DS_COLUMNS:
            if col not in self.df.columns:
                self.df[col] = None
        
        cols = self.DS_COLUMNS + ["text", "split"]
        self.df = self.df[cols]
        
        assert self.df.columns.intersection(self.DS_COLUMNS).shape[0] == len(self.DS_COLUMNS), "There are missing columns"
        
        return self.df
        
    def _download_data(self, download_path):
        # Ensure download path exists
        os.makedirs(download_path, exist_ok=True)

        # Download dataset
        print("Downloading dataset...")
        temp_zip_path = os.path.join(download_path, "temp_dataset.zip")
        handlers.progress_download(self.URL, temp_zip_path)

        # Extract if zip file
        with ZipFile(temp_zip_path, 'r') as zip_file:
            zip_file.extractall(download_path)
        
        # Clean up the temporary zip file
        os.remove(temp_zip_path)
        print("Dataset downloaded and extracted successfully.")

        return download_path
            
class MeddoplaceGazetteer(BenchmarkDataset):
    URL = "https://zenodo.org/records/8403498/files/meddoplace_train+test+gazz+crossmap+multilingual_231003.zip?download=1"
    NAME = "meddoplace_train+test+gazz+crossmap+multilingual"
    DS_COLUMNS = config.DS_COLUMNS
    
    def __init__(self, lang="es", path=None, name=NAME, url=URL, download_if_missing=True):
        super().__init__(lang, name, path, url, download_if_missing)

    def load_data(self):
        '''Load the data from the dataset
        Output: DataFrame with columns: filename, mark, label, off0, off1, span, code, semantic_rel, split, text
        '''
        gaz_path = os.path.join(self.path, "meddoplace_gazetteer/gazetteer_snomed_meddoplace.tsv")
        df = pd.read_csv(gaz_path, sep="\t", dtype=str)
        
        self.df = df
        
        return df
    
    def preprocess_data(self):
        print("preprocessing data...")
        # DS_COLUMNS =  ["filenameid", "mention_class", "span", "code", "sem_rel", "is_abbreviation", "is_composite", "needs_context", "extension_esp"]
        self.df = self.df[config.GZ_COLUMNS]
        
        return self.df
        
    def _download_data(self, download_path):
        # Ensure download path exists
        os.makedirs(download_path, exist_ok=True)

        # Download dataset
        print("Downloading dataset...")
        temp_zip_path = os.path.join(download_path, "temp_dataset.zip")
        handlers.progress_download(self.URL, temp_zip_path)

        # Extract if zip file
        with ZipFile(temp_zip_path, 'r') as zip_file:
            zip_file.extractall(download_path)
        
        # Clean up the temporary zip file
        os.remove(temp_zip_path)
        print("Dataset downloaded and extracted successfully.")

        return download_path

In [13]:
ml = MeddoplaceLoader()
ml.df

preprocessing data...


,filenameid,mention_class,span,code,sem_rel,is_abbreviation,is_composite,needs_context,extension_esp,text,split
0,caso_clinico_medtropical140#469#478,FAC_GEN,albergues,None,None,None,None,None,None,"Hombre 41 años de edad, natural de España que ...",train
1,caso_clinico_medtropical140#642#657,GEO_NOM,bahía de Halong,None,None,None,None,None,None,"Hombre 41 años de edad, natural de España que ...",train
2,caso_clinico_medtropical140#629#634,TRANSPORTE,barco,None,None,None,None,None,None,"Hombre 41 años de edad, natural de España que ...",train
3,caso_clinico_medtropical140#542#551,TRANSPORTE,bicicleta,None,None,None,None,None,None,"Hombre 41 años de edad, natural de España que ...",train
4,caso_clinico_medtropical140#701#710,TRANSPORTE,bicicleta,None,None,None,None,None,None,"Hombre 41 años de edad, natural de España que ...",train
...,...,...,...,...,...,...,...,...,...,...,...
9667,S0120-41572012000500002-1#2366#2377,FAC_GEN,laboratorio,None,None,None,None,None,None,Se presenta el caso de un recién nacido de sex...,test
9668,S1134-80462009000100005-4#190#198,FAC_GEN,farmacia,None,None,None,None,None,None,"Paciente mujer de 70 años de edad, diagnostica...",test
9669,butlleti_camfic_156#515#524,TRANSPORTE,bicicleta,None,None,None,None,None,None,Paciente de 29 años que acude por omalgia izqu...,test
9670,butlleti_camfic_156#1282#1295,DEPARTAMENTO,traumatología,None,None,None,None,None,None,Paciente de 29 años que acude por omalgia izqu...,test


In [14]:
ml_gz = MeddoplaceGazetteer()
ml_gz.df

preprocessing data...


,code,language,term,semantic_tag,mainterm
0,224088008,es,último miembro sobreviviente de la familia,finding,1
1,700850003,es,óvulo para protección de mucosa vaginal,physical object,1
2,464585004,es,óvulo para normalizar el pH vaginal,physical object,1
3,704778003,es,órtesis de mama troncal,physical object,1
4,111042001,es,órgano artificial,physical object,1
...,...,...,...,...,...
27977,108290001,es,radioncología Y/O terapia radiante,procedure,0
27978,108290001,es,radioncología Y/O radioterapia,procedure,1
27979,363087005,es,procedimiento administrativo por discapacidad,procedure,1
27980,386247006,es,mediación de conflicto,procedure,1
